# Datamine DISCAN Process Example

This notebook demonstrates how to configure and run the **`discan`** process wrapper in `dmstudio`.

## Process Description

## DISCAN

The calculation of discriminant functions (Discriminant analysis) will establish a set of rules using geological control groups or training sets which can then be applied to the classification of unknown samples using the [DISCLA](<discla.md>) process.

As a check, the rules as defined by the discriminant functions and group centroids can then be used to reclassify the training sets to see how many mismatches occur.

The linear discriminant function can be considered as a planer separating boundary between the different groups as classified by their input variables (fields) such as CU, PB and ZN. The group centroids are the fixed positions in space locating the centres of each group cluster as defined by the variables.

**Note** : there is a maximum limit of 10 control groups and 45 variables. All individual control groups must also be appended into one file prior to analysis.

### File Handling

The input file (&**IN**) must contain a sample identifier field (@**SAMPID**) and a group identifier field (@**GROUPID**). Data from the different control groups must have a unique identifier common to all samples within the group. All the different control groups must be put into one file using the **APPEND** process before using **DISCAN**. The group identifier will be representative of a geological control group, eg. to identifier them as granite, mineralised zone, background, lithology A or lithology B etc..

There are optional output files for the calculated discriminant functions (&**FUNCTS**), group centroids (&**CENTRDS**) and discriminant scores *&(**SCORES**). If the multiple discriminant functions and group centroids are required to classify unknown data in the **[DISCLA](<discla.md>)** process then the relevant output files must be named. If an output file of the re-classified control data is required then the file &**SCORES** must be used.

If the user wishes to plot maps of the output scores then the scores file can be joined to the original input file using the **[JOIN](<join.md>)** process and defining **SAMPID** as the key field.

### Input Files:

* **in** (Undefined):
  Input file. Must contain group and sample identity fields. A maximum of ten groups are
  evaluated. The file must be sorted on the *GROUPID field.
  Required=Yes

### Output Files:

* **centrds** (Undefined):
  Optional output file containing group centroids identified by the field named in

## GROUPID.

  Required=No

* **functs** (Undefined):
  Optional output file containing discriminant functions.
  Required=No

* **scores** (Undefined):
  Optional output file containing the calculated scores for the individual samples and a
  misclassification field.
  Required=No

### Fields:

* **groupid** (Alphanumeric : IN):
  Compulsory group identifier field contained in input file IN.
  Default=Undefined
  Required=Yes

* **sampid** (Any : IN):
  Compulsory sample identifier field contained in input file IN.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **resum**:

* **Options** ((0): Do not print summary statistics.; 1: Do print summary statistics.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **primat**:

* **Options** ((0): Option to display sums of squares matrices. Set to 1 for matrices to be):
  displayed.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **prisco**:

* **Options** ((0): Option to display group scores. Set to 1 for scores to be displayed.):

* **Note** (do not use for large files.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('discan')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute discan
print("Running discan...")
dm_cmd.discan(
    in_i='t_assays',  # required
    functs_o=['t_discan_out'],  # required
    scores_o=['t_discan_out'],  # required
    groupid_f='optional',  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # centrds_o=['t_discan_out'],  # optional
    # resum_p=0,  # optional
    # primat_p=0,  # optional
    # prisco_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("discan execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_discan_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")